# Fase 3: Calidad y Capacidad de Respuesta por Tecnología

En este cuaderno evaluamos la calidad de las preguntas y la capacidad de respuesta de la comunidad para las 20 tecnologías principales en Stack Overflow. Buscamos respuestas claras a través de "Data Storytelling", utilizando procesamiento perezoso de datos mediante Polars y reutilizando funciones base previamente construidas.

In [23]:
import polars as pl
import plotly.express as px
import sys
import os

# Importar módulos personalizados del proyecto
sys.path.append(os.path.abspath('../src/'))
from metrics import tasa_resolucion, top_n_por_grupo
from survey_utils import pearson_correlation

## 1. Preparación del Dataset Maestro

Realizamos la carga perezosa de los archivos `.parquet`. Cruzamos las preguntas con los tags e identificamos las 20 tecnologías con mayor volumen para centrar nuestro análisis en las comunidades más relevantes. Todo el procesamiento se evalúa perezosamente hasta la llamada de `.collect()`.

In [24]:
# Rutas de entrada a los datos limpios
path_questions = '../data/datos_procesados/dim_questions.parquet'
path_tags = '../data/datos_procesados/fact_question_tags.parquet'

# Inicializamos los LazyFrames (y aplicamos el rename de Tags a Tag al vuelo)
lf_questions = pl.scan_parquet(path_questions)
lf_tags = pl.scan_parquet(path_tags).rename({"Tags": "Tag"})

# Determinamos el Top 20 de tecnologías por volumen de forma lazy
top_20_tags_lf = (
    lf_tags
    .group_by('Tag')
    .agg(pl.len().alias('Count'))
    .sort('Count', descending=True)
    .head(20)
    .select('Tag')
)

# Inner join entre preguntas y tags, y filtramos por el Top 20 usando otro inner join perezoso
df_master = (
    lf_questions
    .join(lf_tags, on='Id', how='inner') # <-- Corregido: ambas tablas usan 'Id'
    .join(top_20_tags_lf, on='Tag', how='inner')
    .collect()
)

print(f"Dimensiones del dataset maestro: {df_master.shape}")

Dimensiones del dataset maestro: (13500688, 8)


## 2. Distribución del Score (Justificación Metodológica)

Visualizaremos cómo se distribuyen las puntuaciones (Score) en Stack Overflow. Esta variable exhibe un fuerte sesgo a la derecha, por lo que usamos un Boxplot y acotamos el eje X. Esto justifica metodológicamente por qué evaluamos la "calidad promedio" a través de la **mediana** y no de la media.

In [18]:
# 1. Cálculo de la mediana global para trazar una línea de referencia (sobre toda la data)
mediana_global = df_master['Score'].median()

# 2. Ordenar los tags por la mediana de forma ascendente para mejorar la narrativa visual
tags_ordenados = (
    df_master.group_by('Tag')
    .agg(pl.col('Score').median())
    .sort('Score')
)['Tag'].to_list()

# 3. Muestreo aleatorio representativo para evitar el colapso de memoria del IDE
# 300,000 filas son más que suficientes para mantener la precisión estadística de los cuartiles
df_visualizacion = (
    df_master.select(['Tag', 'Score'])
    .sample(n=300_000, seed=42) # Semilla 42 para reproducibilidad
    .to_pandas()
)

# 4. Visualización con Plotly
fig_box = px.box(
    df_visualizacion,
    x='Score',
    y='Tag',
    category_orders={'Tag': tags_ordenados},
    title='<b>Distribución del Score por Tecnología</b><br><sup>El 75% de las preguntas tienen puntuaciones bajas (sesgo a la derecha), justificando el uso de la mediana.</sup>',
    labels={'Score': 'Puntuación (Score)', 'Tag': 'Tecnología'}
)

# Ocultar la leyenda redundante y acotar el eje X para esconder los outliers virales extremos
fig_box.update_layout(showlegend=False, xaxis_range=[-3, 15])
fig_box.add_vline(x=mediana_global, line_dash="dash", line_color="red", annotation_text="Mediana Global")

fig_box.show()

## 3. Tasa de Resolución Comunitaria

Medimos la efectividad y disposición de cada comunidad para ayudar. Calculamos el porcentaje de preguntas que reciben al menos una respuesta (`AnswerCount >= 1`). Reutilizamos nuestra función modular `tasa_resolucion` de la carpeta `src`.

In [25]:
# Usamos la función del paquete src para calcular las métricas
df_resolucion = tasa_resolucion(
    df_master, 
    tag_col='Tag', 
    answer_col='AnswerCount',
    closed_col='es_cerrada'
)

# Convertimos el % a formato decimal (0-1) para que Plotly aplique text_auto='.1%' correctamente
df_plot_res = df_resolucion.with_columns(
    (pl.col('Tasa_Respuesta_Pct') / 100).alias('Tasa_Proporcion')
).to_pandas()

# Gráfico de barras horizontales
fig_bar = px.bar(
    df_plot_res,
    x='Tasa_Proporcion',
    y='Tag',
    orientation='h',
    text_auto='.1%',
    title='<b>Tasa de Resolución por Comunidad</b><br><sup>Porcentaje de preguntas que reciben al menos una respuesta válida.</sup>',
    labels={'Tasa_Proporcion': '', 'Tag': 'Tecnología'}
)

# Ocultamos el eje X por redundancia y ordenamos visualmente de mayor a menor
fig_bar.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_visible=False,
    plot_bgcolor='rgba(0,0,0,0)'
)

fig_bar.show()

## 4. Cuadrante de Calidad Comunitaria (Bubble Chart)

Evaluamos la "salud" y efectividad de las distintas comunidades tecnológicas. Para evitar el ruido visual de miles de puntos individuales, agrupamos los datos para calcular la **mediana de visualizaciones** (eje X) y la **mediana de puntuación** (eje Y) por cada tecnología. 

El tamaño de la burbuja representa el volumen total de preguntas en la plataforma. Esto nos permite dividir el panorama en un cuadrante de negocio:
* **Alta Popularidad + Alta Calidad:** Comunidades maduras y altamente resolutivas.
* **Alta Popularidad + Baja Calidad:** Tecnologías con mucho tráfico, pero donde cuesta encontrar respuestas de gran valor.
* **Baja Popularidad + Alta Calidad:** Nichos especializados con comunidades muy dedicadas.

In [30]:
# Opción B: Agrupamos por tecnología para ver el bosque, no los árboles
df_cuadrante = (
    df_master
    .group_by('Tag')
    .agg([
        pl.col('ViewCount').median().alias('Mediana_Vistas'),
        pl.col('Score').median().alias('Mediana_Score'),
        pl.col('Id').len().alias('Volumen_Total') # Para el tamaño de la burbuja
    ])
    .to_pandas()
)

fig_cuadrante = px.scatter(
    df_cuadrante,
    x='Mediana_Vistas',
    y='Mediana_Score',
    size='Volumen_Total',
    color='Tag',
    text='Tag',
    title='<b>Cuadrante de Calidad Comunitaria</b><br><sup>¿Qué tecnología genera el contenido más útil y visto en promedio?</sup>',
    labels={'Mediana_Vistas': 'Mediana de Vistas', 'Mediana_Score': 'Mediana de Puntuación'}
)

# Estética para las burbujas y textos
fig_cuadrante.update_traces(
    textposition='top center', 
    marker=dict(line=dict(width=1, color='DarkSlateGrey'))
)
fig_cuadrante.update_layout(template='plotly_white', showlegend=False)
fig_cuadrante.show()

## 5. Top 5 Preguntas (Entregable para la API y Dashboard)

Finalmente, identificamos las 5 mejores preguntas históricas para cada una de las tecnologías. Generamos la URL canónica y exportamos el entregable en `.parquet` listo para el Dashboard final.

In [32]:
# Obtenemos las Top 5 por Tag
df_top_5 = top_n_por_grupo(df_master, n=5, tag_col='Tag', score_col='Score')

# Ingeniería de Características: URL directa a Stack Overflow
df_top_5 = df_top_5.with_columns(
    ('https://stackoverflow.com/q/' + pl.col('Id').cast(pl.Utf8)).alias('url')
)

# Selección de las columnas entregables
df_final_export = df_top_5.select([
    'Tag', 'Id', 'Score', 'ViewCount', 'AnswerCount', 'url'
])

# Exportar el archivo final para su consumo
output_path = '../data/datos_procesados/top_questions.parquet'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_final_export.write_parquet(output_path)

print("✅ Entregable 'top_questions.parquet' guardado exitosamente.\n")
print("Muestra de las mejores preguntas evaluadas:")
print(df_final_export.head(10))

✅ Entregable 'top_questions.parquet' guardado exitosamente.

Muestra de las mejores preguntas evaluadas:
shape: (10, 6)
┌─────────┬──────────┬───────┬───────────┬─────────────┬─────────────────────────────────┐
│ Tag     ┆ Id       ┆ Score ┆ ViewCount ┆ AnswerCount ┆ url                             │
│ ---     ┆ ---      ┆ ---   ┆ ---       ┆ ---         ┆ ---                             │
│ str     ┆ i64      ┆ i64   ┆ i64       ┆ i64         ┆ str                             │
╞═════════╪══════════╪═══════╪═══════════╪═════════════╪═════════════════════════════════╡
│ android ┆ 45940861 ┆ 1715  ┆ 1511426   ┆ 37          ┆ https://stackoverflow.com/q/45… │
│ android ┆ 34353220 ┆ 1235  ┆ 618014    ┆ 55          ┆ https://stackoverflow.com/q/34… │
│ android ┆ 38200282 ┆ 923   ┆ 631960    ┆ 23          ┆ https://stackoverflow.com/q/38… │
│ android ┆ 66980512 ┆ 897   ┆ 717782    ┆ 46          ┆ https://stackoverflow.com/q/66… │
│ android ┆ 29041027 ┆ 855   ┆ 491193    ┆ 16          ┆ http